# Entrenamiento sin ruido analógico

In [ ]:
!nvidia-smi
!for pid in $(nvidia-smi --query-compute-apps=pid --format=csv,noheader); do ps -o user=,pid=,etime=,cmd= -p $pid; done

In [ ]:
%pip install ultralytics==8.4.173 albumentations==2.0.8

## Imports

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display
from ultralytics import YOLO

## Rutas y parámetros

In [ ]:
# Carpeta que contiene datasets/: la del notebook en el servidor o la raíz del repo
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()), Path.cwd())
DATA = REPO / "datasets/floor-plan-cad-yolo/floor-plan-cad-yolo.yaml"
RUNS = REPO / "runs"

MODEL = "yolo26m-seg.pt"
IMGSZ = 1024
BATCH = 2
EPOCHS = 120
SEED = 42
DEVICE = 0
PATIENCE = 20
RESUME = False  # True retoma la corrida cortada desde su last.pt
NAME = f"{Path(MODEL).stem}-no-noise"  # distinto de la corrida con ruido analógico
RUN_DIR = RUNS / NAME  # el entrenamiento la actualiza si Ultralytics usa otro nombre (p. ej. yolo26m-seg-no-noise-2)

assert DATA.exists(), f"No existe el dataset: {DATA}"

## Entrenamiento

> Mismos parámetros que `train_segmentation_model.ipynb`, pero sin Albumentations: sirve para medir cuánto aporta el ruido analógico.

In [ ]:
if RESUME:
    # Usa los mismos argumentos guardados en el checkpoint
    model = YOLO(RUN_DIR / "weights" / "last.pt")
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(
        data=str(DATA), imgsz=IMGSZ, batch=BATCH, epochs=EPOCHS,
        seed=SEED, deterministic=True, device=DEVICE, patience=PATIENCE,
        project=str(RUNS), name=NAME,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.4,  # planos en gris: sin tinte, solo brillo
        degrees=3.0,                      # escaneo levemente torcido
        flipud=0.5, fliplr=0.5,           # el plano no tiene arriba ni abajo
        mosaic=1.0, scale=0.5,
        augmentations=[],  # sin Albumentations, tampoco las que Ultralytics agrega por defecto
    )
RUN_DIR = Path(model.trainer.save_dir)
print(RUN_DIR)

## Evaluación en test

In [ ]:
best = YOLO(RUN_DIR / "weights" / "best.pt")
metrics = best.val(data=str(DATA), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                   project=str(RUN_DIR), name="test")

## Métricas de test

In [ ]:
rows = [("Box", metrics.box.map50, metrics.box.map),
        ("Seg", metrics.seg.map50, metrics.seg.map)]
display(Markdown(
    f"**Test · {NAME} · imgsz={IMGSZ}**\n\n| | mAP50 | mAP50-95 |\n|---|---|---|\n"
    + "\n".join(f"| {name} | {m50:.3f} | {m:.3f} |" for name, m50, m in rows)))